# South Kensington DIGIT Runner

Rebuilds South Kensington building geometry from raw OSM footprints at **2 m**,
runs DIGIT on a GPU-friendly derived grid, saves PNG outputs, and exports a time animation.


The geometry is too large for github. Please visit:

https://jupyter.cx3.rcs.ic.ac.uk/hub/user-redirect/lab/tree/Velocity/hpc_jupyter_bundle/assets/data/imperial_10m/raw/osm_buildings.json

and

https://jupyter.cx3.rcs.ic.ac.uk/hub/user-redirect/lab/tree/Velocity/hpc_jupyter_bundle/assets/data/imperial_10m/static

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

required = ["matplotlib", "IPython", "PIL"]
missing = []
for name in required:
    spec_name = "PIL" if name == "PIL" else name
    if importlib.util.find_spec(spec_name) is None:
        missing.append("pillow" if name == "PIL" else name)

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", *missing])
else:
    print("Notebook dependencies already available.")

bundle_root = Path.cwd()
bundle_root

In [ ]:
from pathlib import Path

import torch

from south_kensington_jupyter import (
    SouthKensingtonConfig,
    extract_embedded_height_voxels,
    plot_3d_velocity_volume,
    run_pipeline,
    save_3d_velocity_view,
    show_vertical_section_animation_from_results,
    save_summary_and_arrays,
    show_animation_from_results,
)

print("torch", torch.__version__)
print("cuda_available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu_name", torch.cuda.get_device_name(0))

In [ ]:
config = SouthKensingtonConfig(
    geometry_resolution_m=2.0,
    model_resolution_m=4.0,
    timesteppings=70,
    time_levels=(0, 20, 45, 70, 95),
    slice_indices=(1, 4, 20),
    animation_slice_indices=(1, 4, 20),
    animation_slice_idx=4,
    animation_stride=4,
    output_dir=str(Path("outputs") / "south_kensington_digit_gpu_2m"),
)
config

In [ ]:
results = run_pipeline(config)
saved = save_summary_and_arrays(results)
saved

In [ ]:
summary_path = Path(saved["summary_path"])
print(summary_path.read_text())

In [ ]:
show_animation_from_results(results, slice_idx=1)

In [ ]:
show_animation_from_results(results, slice_idx=4)

In [ ]:
show_animation_from_results(results, slice_idx=20)

In [ ]:
view_3d_path = save_3d_velocity_view(results, timestep_idx=-1, slice_indices=(1, 4, 20))
view_3d_path

In [ ]:
show_vertical_section_animation_from_results(results)

In [ ]:
output_dir = Path(saved["summary_path"]).resolve().parent
root_dir = bundle_root.resolve()
listed = []
for path in sorted(output_dir.iterdir()):
    try:
        shown = path.resolve().relative_to(root_dir)
    except ValueError:
        shown = path.resolve()
    listed.append(str(shown))
listed